# Lab 1 — Your First ML Pipeline
**Machine Learning for the Natural Sciences**

Welcome! In this lab you'll build a complete machine learning pipeline from scratch
using the Palmer Penguins dataset. By the end, you'll have a working species classifier
and understand every step of the ML workflow.

**What you'll learn:**
- Loading and inspecting data with pandas
- Exploratory data analysis (EDA) with seaborn
- Preparing data for scikit-learn
- Training a K-Nearest Neighbors classifier
- Evaluating with accuracy and a confusion matrix

**The 7-Step Pipeline:**
1. Load → 2. Explore → 3. Clean → 4. Split → 5. Train → 6. Evaluate → 7. Iterate

---

## Step 0: Imports
These are the libraries you'll use in almost every lab this quarter.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

# Reproducibility
RANDOM_STATE = 42

## Step 1: Load the Data
Palmer Penguins is built into seaborn — no file download is needed.
It contains measurements of 344 penguins from three species
(Adelie, Chinstrap, Gentoo) on three islands in Antarctica.

In [2]:
df = sns.load_dataset("penguins")
print(f"Shape: {df.shape}")
df.head()

Shape: (344, 7)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female


## Step 2: Explore
Before building any models, **look at your data**. This is the most important
habit in data science.

In [ ]:
# Basic info — data types, non-null counts
df.info()

In [ ]:
# How many missing values per column?
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal rows: {len(df)}")

In [ ]:
# Class distribution — are the species balanced?
print("Species counts:")
print(df["species"].value_counts())

In [ ]:
# Summary statistics for numeric columns
df.describe()

In [ ]:
# Pairplot — the single most useful EDA plot for tabular data
# Look for: which features separate species? Where is there overlap?
sns.pairplot(df, hue="species", diag_kind="hist")
plt.suptitle("Palmer Penguins — Pairplot by Species", y=1.02)
plt.tight_layout()
plt.show()

### 🔍 Your Turn: EDA Observations
**TODO:** In your own words, answer these questions based on the pairplot:

1. Which two features seem to separate Gentoo from the other species best?

   *Your answer:*

2. Which pair of species has the most overlap (hardest to tell apart)?

   *Your answer:*

3. Are there any features that do NOT help distinguish species?

   *Your answer:*

## Step 3: Clean and Prepare
Scikit-learn needs numeric arrays with no missing values.

In [ ]:
# For now, we'll drop rows with missing values.
# (We'll learn smarter imputation strategies in Week 2.)
df_clean = df.dropna()
print(f"Rows before: {len(df)}, after: {len(df_clean)}, dropped: {len(df) - len(df_clean)}")

In [ ]:
# Select numeric features for our model
feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df_clean[feature_cols]
y = df_clean["species"]

print(f"Features shape: {X.shape}")
print(f"Target classes: {y.unique()}")

In [ ]:
# Encode species as integers (scikit-learn convention)
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## Step 4: Train/Test Split
We hold out 20% of the data as a "test set" — the model **never** sees this
during training. This is how we check whether the model learned general
patterns vs. just memorizing the training data.

Think of it this way: the training set is the textbook, and the test set is the exam.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

## Step 5: Train the Model
K-Nearest Neighbors (KNN) is the simplest classifier to understand.
When it gets a new penguin, it looks at the `k` closest penguins in the
training set and takes a majority vote on the species.

In [ ]:
# Create and train the model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

# Make predictions on the test set
y_pred = knn.predict(X_test)

## Step 6: Evaluate

In [ ]:
# Overall accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

In [ ]:
# Confusion matrix — this tells you WHERE the model is making mistakes
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(cmap="Blues")
plt.title("KNN Confusion Matrix — Penguins")
plt.tight_layout()
plt.show()

### 🔍 Your Turn: Evaluate
**TODO:** Look at the confusion matrix and answer:

1. Which species does the model classify most accurately?

   *Your answer:*

2. Are there any species the model confuses with each other? Which ones?

   *Your answer:*

3. Does this match what you saw in the pairplot?

   *Your answer:*

## Step 7: Iterate
The beauty of ML is rapid experimentation. Let's try changing things.

### Experiment 1: Does scaling help?
KNN uses distances — features on different scales can dominate.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
y_pred_scaled = knn_scaled.predict(X_test_scaled)
acc_scaled = accuracy_score(y_test, y_pred_scaled)

print(f"Without scaling: {accuracy:.3f}")
print(f"With scaling:    {acc_scaled:.3f}")

### Experiment 2: What's the best value of k?
**TODO:** Fill in the loop below to test different values of k.

In [ ]:
k_values = range(1, 21)
accuracies = []

for k in k_values:
    # TODO: Create a KNN model with k neighbors, fit on scaled training data,
    # predict on scaled test data, and append the accuracy to the list.
    # (Hint: it's the same 3 lines from Step 5, but with k instead of 5)

    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, model.predict(X_test_scaled))
    accuracies.append(acc)

# Plot accuracy vs. k
plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracies, marker="o", linewidth=2, markersize=5)
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("Test Accuracy")
plt.title("KNN: Accuracy vs. k")
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(accuracies)]
print(f"Best k = {best_k} with accuracy = {max(accuracies):.3f}")

### 🔍 Your Turn: Reflect
**TODO:** Answer these wrap-up questions:

1. Did scaling improve accuracy? Why might that be? (Hint: look at the
   ranges of the four features — are they all on the same scale?)

   *Your answer:*

2. What was the best k you found? What happens when k is very small (1)?
   Very large (20)?

   *Your answer:*

3. In your own words, describe how KNN makes a prediction.

   *Your answer:*

---
## What to Submit
1. A PDF with your answers to all the **🔍 Your Turn** sections
2. A link to your completed Colab notebook
3. Keep it short — this should be a 1–2 page document.

## What's Next
In **Lab 2** we'll move from classification to regression, predicting penguin
body mass from the other measurements. We'll also start exploring feature
engineering — creating new, more informative features from the ones we have.